# Cross-simulator validation of BioModels

_Investigation `biomodels-cross-simulator-validation` — coder reproduction notebook._

**Question.** Do independent SBML simulators reproduce curated BioModels, and how well do they agree with each other and with the BioSimulators reference results?

A process-bigraph workspace that loads BioModels, runs them under multiple simulators (and the BioSimulators SED-ML references), and scores agreement. Both the interactive report and the zarr results archive are outputs of the batch-compare-biomodels composite.

---

This notebook re-runs each study with the workspace's own process-bigraph protocol and renders its figures. The text states the **question and parameters** only — the figures produced by each run are the results. Set `RERUN = False` in the setup cell to render the committed `runs.db` without re-simulating.


In [ ]:
"""Self-contained reproduction of this investigation.

Generated by vivarium-dashboard (notebook_export). Each study below is re-run
live with the workspace's own process-bigraph protocol and its figures are
rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/Users/eranagmon/code/pbg-biomodels--zarr-export').is_dir():
    REPO = Path('/Users/eranagmon/code/pbg-biomodels--zarr-export')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# --- standard process-bigraph protocol: register the workspace's Core ---
from viva_biomodels.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

## Study: `cross-simulator-comparison`

**Question.** Do independent SBML simulators (COPASI, Tellurium, simbio) agree with one another and with the BioSimulators SED-ML reference results on curated BioModels, and where do they diverge?


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `batch-compare` | `viva_biomodels.composites.batch_compare_biomodels` | 5 | biomodel_ids=['BIOMD0000000001', 'BIOMD0000000002', 'BIOMD0000000003'], simulators=['copasi', 'tellurium', 'simbio'], reference_results_dir=datasets/biosimulators_sedml_results, zarr_out=out/cross_simulator_comparison.zarr |
| `batch-compare` | `viva_biomodels.composites.batch_compare_biomodels` | 5 | biomodel_ids=['BIOMD0000000001', 'BIOMD0000000002', 'BIOMD0000000003'], simulators=['copasi', 'tellurium', 'simbio'], reference_results_dir=datasets/biosimulators_sedml_results, zarr_out=out/cross_simulator_comparison.zarr |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_biomodels.composites.batch_compare_biomodels`** — `spec_viva_biomodels_composites_batch_compare_biomodels` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_viva_biomodels_composites_batch_compare_biomodels = load_spec(REPO / 'viva_biomodels/composites/viva_biomodels.composites.batch_compare_biomodels.composite.yaml')
describe_spec(spec_viva_biomodels_composites_batch_compare_biomodels)

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: cross-simulator-comparison ===
STUDY = 'cross-simulator-comparison'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Runtime knobs — edit freely. STEPS = number of composite steps;
# INTERVAL = global dt filling ${interval} placeholders (a per-process
# interval pinned in the edit cell above takes precedence).
STEPS_batch_compare = 5
INTERVAL_batch_compare = 0.1
STEPS_batch_compare = 5
INTERVAL_batch_compare = 0.1

if RERUN:
    with quiet():  # the sim prints per-step progress; keep it out of the notebook
        # Generic process-bigraph protocol (no workspace runner detected):
        from viva_superpowers.composite_spec import build_composite_from_spec
        comp = build_composite_from_spec(spec_viva_biomodels_composites_batch_compare_biomodels, {'interval': INTERVAL_batch_compare}, core=core)
        comp.run(STEPS_batch_compare)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_viva_biomodels_composites_batch_compare_biomodels, {'interval': INTERVAL_batch_compare}, core=core)
        comp.run(STEPS_batch_compare)  # writes the composite's declared emitter
    print(f'ran 2 simulation(s) -> {RUNS_DB}')
else:
    print("RERUN=False — rendering committed", RUNS_DB)